# Assignment 3: Assemble IR and LM for RAG

## Overview
Assemble the BM25 retriever from Assignment 1 and the text generator from Assignment 2 to build a RAG system. Test it with open-form questions on the Cranfield collection. This assignment covers four tasks: building the RAG system, selecting the best generator, understanding how retrieval affects output quality, and diagnosing where the pipeline succeeds or fails. All evaluations in this assignment are qualitative. In your analysis, do not just describe what the model outputs, but engage with why it behaves the way it does and what it reveals about the system.

## Requirements
- Make sure all cells have been run and outputs are visible. Queries and model responses **must be displayed** in the notebook output.
- Implement each task in the cells marked **TODO** and answer questions marked **TODO**

## Deadline
**May 1, 23:59 CET**

## Submission
- Submit only the .ipynb file. Add your name to the filename.

In [180]:
# Load imports
from __future__ import annotations

import math
import re
from collections import Counter, defaultdict
from dataclasses import dataclass
from typing import Dict, Iterable, List, Tuple, Optional

import numpy as np
import torch
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from transformers import AutoTokenizer, AutoModelForCausalLM
import ir_datasets


## Load the models and tokenizer

In [181]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen1.5-0.5B")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Replace these paths with your actual Assignment 2 checkpoints
model_full = AutoModelForCausalLM.from_pretrained("../Assignment_2/ckpt-full")

model_completion = AutoModelForCausalLM.from_pretrained("../Assignment_2/ckpt-completion")

print("Both models loaded.")

Loading weights: 100%|██████████| 291/291 [00:00<00:00, 18902.04it/s]
The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Loading weights: 100%|██████████| 291/291 [00:00<00:00, 7922.87it/s]
The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Both models loaded.


## Task 1: Build a RAG System

### Task overview
1. Implement a `BM25` class (ported from Assignment 1) and build an index over the Cranfield collection.
2. Implement a `RAGSystem` class that ties retrieval and generation together.




### Task 1.1: BM25 Retriever

Port your `BM25` implementation from Assignment 1, keeping only the attributes and methods you require.

In [182]:
@dataclass
class BM25:
    """

    Brute-force scoring over all documents (no inverted index).
    """
    def __init__(self, k1: float = 1.2, b: float = 0.75, remove_stopwords: bool = True):
    
        # --- BM25 Hyperparameters ---
        self.k1: float = k1                    # term frequency saturation
        self.b: float = b                      # document length normalization
        self.remove_stopwords: bool = remove_stopwords
        
        # Initializations for tokenization
        self._stemmer = PorterStemmer()      # work stemming
        self._stopwords = set(stopwords.words("english")) if self.remove_stopwords else set() # stopwords set
        
        # --- Corpus Statistics (populated by build()) ---
        self.N: int = 0                       # total number of documents
        self.avgdl: float = 0.0               # average document length
        self.df: Dict[str, int] = {}          # term -> number of docs containing term
        self.doc_len: Dict[str, int] = {}     # doc_id -> document length
        self.doc_tf: Dict[str, Counter] = {}  # doc_id -> term frequency Counter
        self.docs_store: Dict[str, Dict[str, str]] = {}  # doc_id -> {"title": ..., "text": ...}

    def tokenize(self, text: str) -> List[str]:
        """
        Convert raw text into a list of processed tokens.

        Requirements:
        1. Lowercase everything         e.g.  "UZH"                       -> ["uzh"]
        2. Extract alphanumeric words   e.g.  "hello, world 2026!"        -> ["hello", "world", "2026"]
        3. Remove stopwords             e.g.  "the document has a title"  -> ["document", "title"]
        4. Apply Porter stemming        e.g.  "working worked works"      -> ["work", "work", "work"]
        5. Drop empty tokens            e.g.  " retrieval "               -> ["retrieval"]

        Args:
            text: raw input string
        Returns:
            List of processed token strings

        Hints:
            - Perform text normalization using Python string methods.
            - Use the `re` module to extract alphanumeric tokens.
            - For stemming and stopword removal, use the initialized stemmer and stopword list from the BM25 class constructor.

        """

        text = text.lower()
        text = re.sub(r'[\W_]', ' ', text)
        tokenized_text: list[str] = [word for word in text.split() if word not in self._stopwords]
        return [self._stemmer.stem(word) for word in tokenized_text]
    
    def build(self, docs_iter: Iterable) -> None:
        """
        Iterate over all documents and precompute corpus statistics needed for BM25.

        Args:
            docs_iter: iterable of Cranfield documents, each a namedtuple with fields:
                    (doc_id, title, text, author, bib)
        
        Requirements: 
            After calling build(), the following must be populated:
                self.N        : int                      — total number of documents
                self.avgdl    : float                    — average document length
                self.df       : Dict[str, int]           — term -> number of docs containing term
                self.doc_len  : Dict[str, int]           — doc_id -> document length
                self.doc_tf   : Dict[str, Counter]       — doc_id -> term frequencies per document
                self.docs_store: Dict[str, Dict[str, str]] — doc_id:str -> {"title": ..., "text": ...} (raw title & text per document)

        Note:
            Per-document steps (repeat for each document):
                1. Extract title and text safely
                2. Concatenate title and text, then tokenize using self.tokenize()
                3. Compute term frequencies  → store in self.doc_tf
                4. Compute document length → store in self.doc_len
                5. Store raw title and text → store in self.docs_store

            Corpus-level steps (after all documents):
                6. Count how many documents each term appears in → store in self.df
                (each term is counted once per document, not once per occurrence)
                7. Compute self.avgdl and self.N

        """
        df_counter = defaultdict(int)
        total_len = 0

        for doc in docs_iter:
            self.N += 1

            title = doc.title if doc.title is not None else ""
            text = doc.text if doc.text is not None else ""
            tokens = self.tokenize(" ".join([title, text]))

            tf = Counter(tokens)
            doc_id = (doc.doc_id)

            self.doc_tf[doc_id] = tf
            self.doc_len[doc_id] = len(tokens)
            self.docs_store[doc_id] = {"title": title, "text": text}

            total_len += len(tokens)

            for term in tf:
                df_counter[term] += 1

        self.df = dict(df_counter)
        self.avgdl = total_len / self.N if self.N > 0 else 0.0

    def idf(self, term: str) -> float:
        """
        Compute Inverse Document Frequency with standard smoothing.

        Args:
            term: a single token str

        Returns:
            float — IDF score for the term

        Formula:
            log( (N - df + 0.5) / (df + 0.5) + 1 )

        """
        df = self.df.get(term, 0)
        return math.log((self.N-df+0.5)/(df+0.5)+1)
    
    def score(self, query_tokens: List[str], doc_id: str) -> float:
        """
        Compute BM25 score for a query against a single document.

        Args:
            query_tokens: list of tokenized query terms
            doc_id: document to score against

        Returns:
            float — BM25 score for this query-document pair

        Formula (sum over query terms):
            score(q, d) = sum over t in q [ IDF(t) * (f * (k1 + 1)) / (f + k1 * (1 - b + b * dl/avg_dl)) ]

        where:
            q    = query
            d    = document
            t    = a query term
            f    = term frequency of t in document
            dl   = length of the document
            k1   = self.k1
            b    = self.b

        Hints:
            - skip terms with frequency 0
            - return 0.0 for empty documents
            - sum over query terms (you can keep duplicates, but set() is common and stable for students)
        """

        dl = self.doc_len.get(doc_id, 0)
        if dl == 0: return 0

        total = 0
        for term in set(query_tokens):
            tf = self.doc_tf[doc_id].get(term, 0)
            if tf == 0: continue
            total += self.idf(term) * (tf*(self.k1+1)) / (tf+self.k1*(1-self.b+self.b*dl/self.avgdl))
        return total
    
    def retrieve(self, query_text: str, k: int = 10) -> List[Tuple[str, float]]:
        """
        Return the top-k documents for a given query, ranked by BM25 score.
        Brute-force over all documents.
        
        Args:
            query_text: raw query string
            k:          number of documents to return

        Returns:
            List of (doc_id, score) tuples sorted by score descending

        Hints:
            - tokenize the query first using self.tokenize()
            - iterate over all documents in self.doc_tf.keys()
            - use self.score() to compute the BM25 score for each document
            - only keep documents with a non-zero score
        """
        score_tuple_list = []

        query_tokenized = self.tokenize(query_text)
        for doc_id in self.doc_tf.keys(): 
            score = self.score(query_tokenized, doc_id)
            if score > 0:
                score_tuple_list.append((doc_id, score))
        score_tuple_list.sort(key=lambda x: x[1], reverse=True)
        return score_tuple_list[:k]
    
    def evaluate(
        self,
        queries_iter: Iterable,
        qrels_iter: Iterable,
        k: int = 10,
        metrics: Optional[List[str]] = None,
    ) -> Dict[str, float]:
        """
        Compute evaluation metrics averaged over all queries.

        Args:
            queries_iter : iterable of queries (each has .query_id and .text)
            qrels_iter   : iterable of relevance judgments (each has .query_id, .doc_id, .relevance)
            k            : number of top documents to evaluate
            metrics      : list of metrics to compute, e.g. ["recall", "precision", "mrr", "ndcg"]
                        if None, compute all four

        Returns:
            Dict mapping metric name -> average score across all queries

        Note:
            Relevance:
                - treat relevance <= 0 as non-relevant (0)
                - treat relevance  > 0 as relevant (1)

            Metrics:
                Recall@k    = (number of relevant docs in top-k) / (total relevant docs for query)

                Precision@k = (number of relevant docs in top-k) / k

                MRR@k       = 1 / (rank of first relevant doc in top-k)
                            0 if no relevant doc found

                nDCG@k      = DCG@k / IDCG@k
                            DCG@k  = sum over rank i: rel_i / log2(i + 2)    (i is 0-indexed)
                            IDCG@k = DCG of the ideal ranking (sort all relevant docs by relevance desc)

        Hints:
            - define two helper functions inside evaluate():
                * normalize_relevance(rel) to convert relevance scores to 0 or 1 (binary)
                * dcg(rels) to compute DCG for a ranked relevance list
            - build a qrels dict: qrels[query_id][doc_id] = normalized relevance
            - skip queries that have no relevance judgments
            - use self.retrieve(query_text, k) to get ranked (doc_id, score) pairs
            - for duplicate qrel entries keep the maximum relevance
        """

        if metrics is None:
            metrics = ["recall", "precision", "mrr", "ndcg"]

        def normalize_relevance(rel):
            return 1 if rel > 0 else 0

        def dcg(rels):
            return sum(rel / math.log2(i + 2) for i, rel in enumerate(rels))

        qrels = defaultdict(dict)
        for qrel in qrels_iter:
            query_id = qrel.query_id
            doc_id = qrel.doc_id
            rel = normalize_relevance(qrel.relevance)
            qrels[query_id][doc_id] = max(qrels[query_id].get(doc_id, 0), rel)

        totals = {metric: 0.0 for metric in metrics}
        evaluated_queries = 0

        for query in queries_iter:
            query_id = query.query_id
            if query_id not in qrels:
                continue

            query_qrels = qrels[query_id]
            ranked_docs = self.retrieve(query.text, k)
            ranked_rels = [query_qrels.get(doc_id, 0) for doc_id, _ in ranked_docs]

            relevant_in_top_k = sum(ranked_rels)
            total_relevant = sum(query_qrels.values())

            if "recall" in totals:
                totals["recall"] += relevant_in_top_k / total_relevant if total_relevant > 0 else 0.0

            if "precision" in totals:
                totals["precision"] += relevant_in_top_k / k if k > 0 else 0.0

            if "mrr" in totals:
                reciprocal_rank = 0.0
                for rank, rel in enumerate(ranked_rels, start=1):
                    if rel > 0:
                        reciprocal_rank = 1.0 / rank
                        break
                totals["mrr"] += reciprocal_rank

            if "ndcg" in totals:
                ideal_rels = sorted(query_qrels.values(), reverse=True)[:k]
                ideal_dcg = dcg(ideal_rels)
                totals["ndcg"] += dcg(ranked_rels) / ideal_dcg if ideal_dcg > 0 else 0.0

            evaluated_queries += 1

        if evaluated_queries == 0:
            return {metric: 0.0 for metric in metrics}
        
        return {metric: totals[metric] / evaluated_queries for metric in metrics}
    
    def grid_search(
        self,
        queries_iter_factory,
        qrels_iter_factory,
        k: int = 10,
        k1_values: Tuple = (0.60, 1.20, 1.80),
        b_values: Tuple = (0.25, 0.50, 0.75),
    ) -> List[Dict]:
        """
        Evaluate all combinations of k1 and b, return results as a list of dicts.

        Args:
            queries_iter_factory : callable that returns a fresh queries iterator each time
            qrels_iter_factory   : callable that returns a fresh qrels iterator each time
            k                    : number of top documents to evaluate
            k1_values            : values of k1 to try
            b_values             : values of b to try

        Returns:
            List of dicts, one per (k1, b) combination:
            [
                {"k1": 0.60, "b": 0.25, "recall": ..., "precision": ..., "mrr": ..., "ndcg": ...},
                ...
            ]

        Note:
            - update self.k1 and self.b directly before each self.evaluate() call
            - pass factories/lambdas not iterators as ir_datasets iterators are single-pass,
            so call queries_iter_factory() and qrels_iter_factory() inside the loop to get a fresh iterator each time
        """

        grid_search_list = []

        for k1 in k1_values: 
            for b in b_values:
                self.k1 = k1
                self.b = b
                eval = self.evaluate(queries_iter_factory(), qrels_iter_factory(), k=k)
                grid_search_list.append({
                    "k1": k1,
                    "b": b,
                    "recall": eval.get("recall"),
                    "precision": eval.get("precision"),
                    "mrr": eval.get("mrr"),
                    "ndcg": eval.get("ndcg")
                })

        return grid_search_list

### Build the Cranfield Index

Load the Cranfield corpus with `ir_datasets` and call `bm25.build()`. You may use the best `k1` and `b` hyperparameters from Assignment 1.


In [183]:
# Load the Cranfield corpus and build the BM25 index
dataset = ir_datasets.load("cranfield")
bm25 = BM25(k1=1.8, b=0.75)   # <- use your best k1/b from Assignment 1
bm25.build(dataset.docs_iter())

print(f"Index built: {bm25.N} documents, avg doc length = {bm25.avgdl:.1f} tokens")


Index built: 1400 documents, avg doc length = 102.9 tokens


### Task 1.2: RAGSystem

Implement `RAGSystem` with the following behaviour:

* **`build_prompt(query, retrieved_docs)`** — assemble the prompt using the **same instruction-tuning template** as in Assignment 2 (`format_prompt`). Concatenate the retrieved documents as a single doc to accomodate *k* larger than 1.
* **`generate(query, k, verbose)`** — full RAG pipeline:
  1. Retrieve top-*k* docs via `self.bm25.retrieve()`
  2. Build the prompt with `build_prompt`
  3. Run inference; **strip the prompt prefix** from the decoded output
  4. Return the answer string only
  5. Have a *verbose* flag to return the retrieved docs too. It will be useful for evaluation.
The constructor must accept `model`, `tokenizer`, `bm25`, `k` (default 3), and `max_new_tokens` (default 256).


In [184]:
class RAGSystem:
    """
    Retrieval-Augmented Generation (RAG) pipeline combining BM25 retrieval
    with a fine-tuned causal language model for open-form question answering.

    The pipeline operates in two stages:
      1. Retrieval: the BM25 index is queried to fetch the top-k most
         relevant documents for a given question.
      2. Generation: the retrieved documents are injected into an
         instruction-tuning prompt and passed to the language model, which
         produces a grounded answer.

    Typical usage:
        rag = RAGSystem(model=model, tokenizer=tokenizer, bm25=bm25)
        answer = rag.generate("What causes boundary layer separation?", k=3)
    """

    def __init__(
        self,
        model,
        tokenizer,
        bm25: BM25,
        max_new_tokens: int = 500,
    ):
        """
        Initialise the RAG system.

        Args:
            model          : fine-tuned causal language model used for generation.
            tokenizer      : tokenizer corresponding to *model*
            bm25           : a fully built BM25 instance (``bm25.build()`` must
                             have been called before passing it here).
            max_new_tokens : maximum number of new tokens the model may generate
                             per call to :meth:`generate`.
        """
        self.model = model
        self.tokenizer = tokenizer
        self.bm25 = bm25
        self.max_new_tokens = max_new_tokens
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model.to(self.device)

    def build_prompt(
        self,
        query: str,
        retrieved_docs: list,
        instruction: str | None = None,
    ) -> str:
        """
        Assemble the instruction-tuning prompt used for RAG generation.

        Follows the same three-block template as Assignment 2's ``format_prompt``:
        ``### Instruction``, ``### Input``, ``### Response``.  
        
        Retrieved documents are concatenated (title + body) and inserted as the
        ``Document`` field of the input block, allowing the model to draw on
        multiple passages in a single forward pass.

        **Tip**: If you are unsure what the prompt should look like, inspect a
        few samples from your Assignment 2 test set — the structure used there
        is exactly what this method should replicate (minus the retrieved
        context, which replaces the original document field).

        Args:
            query         : the user question or instruction to be answered.
            retrieved_docs: list of document dicts, each containing ``"title"``
                            and ``"text"`` keys (as stored in ``BM25.docs_store``).
            instruction   : optional override for the ``### Instruction`` block.
                            Defaults to a generic technical-QA instruction when
                            *None*. Look at Test data samples for sample instructions

        Returns:
            Formatted prompt string ready for tokenisation, ending with
            ``"### Response:\\n"`` so the model continues from that position.
        """
        if instruction is None:
            instruction = "Answer the following technical question using only the information in the document."

        _parsed_docs = "\n\n".join(
            f"{(d.get('title') or '').strip()}\n{(d.get('text') or '').strip()}"
            for d in retrieved_docs
        ).strip()

        _first_of = f"### Instruction: {instruction}\n\n"
        _second_of = f"### Input:\nQuestion:\n{query}\n\nDocument:\n{_parsed_docs}\n\n"
        _eof = "### Response:\n"

        _output = _first_of + _second_of + _eof

        return _output

    def generate(
        self,
        query: str,
        k: int = 3,
        verbose: bool = False,
        instruction: str | None = None,
    ) -> str | tuple:
        """
        Run the full RAG pipeline for a single query.

        Steps:
          1. Retrieve the top-k documents from the BM25 index.
          2. Build the instruction-tuning prompt via `build_prompt`.
          3. Tokenise the prompt and run generation with the language model.
          4. Decode only the newly generated tokens (the prompt prefix is stripped).
          5. Return the answer, optionally alongside the retrieved documents.

        Args:
            query   : raw (un-tokenised) user question.
            k       : number of documents to retrieve and include in the prompt.
            verbose : when *True*, returns a ``(answer, retrieved_docs)`` tuple
                      so callers can inspect which passages informed the answer.

        Returns:
            The generated answer string when verbose is False, or a
            ``(answer, retrieved_docs)`` tuple when verbose is True.
        """
        topk_idx = self.bm25.retrieve(query, k=k)
        topk_formatted = [self.bm25.docs_store[id] for (id, _) in topk_idx]
        prompt = self.build_prompt(query, topk_formatted, instruction=instruction)
        prompt = self.tokenizer(prompt, return_tensors="pt").to(self.device)
        prompt_len = prompt["input_ids"].shape[1]

        with torch.no_grad():
            answer = self.model.generate(
                    **prompt,
                    max_new_tokens=self.max_new_tokens,
                    pad_token_id=self.tokenizer.eos_token_id,
                )
            
        answer = answer[0, prompt_len:]
        answer = self.tokenizer.decode(answer, skip_special_tokens=True).strip()
        
        if verbose:
            return answer, topk_formatted
        return answer


### Task 1.3: Check your implementation

Run the cell below to verify that the full RAG pipeline works end-to-end.  
**Note:** this cell requires a loaded model and tokenizer — run it after you have loaded at least one checkpoint in Problem 2.

In [185]:
# Full RAG test
# Replace model / tokenizer with whichever checkpoint you loaded first.
SAMPLE_QUERY = "what are the structural and aeroelastic problems associated with flight of high speed aircraft"

rag_test = RAGSystem(model=model_completion, tokenizer=tokenizer, bm25=bm25)

answer, docs = rag_test.generate(SAMPLE_QUERY, k=1, verbose=True)

print("=== Retrieved documents ===")
for i, doc in enumerate(docs, 1):
    title = doc.get("title", "").strip() or "(no title)"
    snippet = doc.get("text", "").replace("\n", " ")
    print(f"[{i}] {title}\n    {snippet}...\n")

print("=== Generated answer ===")
print(answer)

=== Retrieved documents ===
[1] some structural and aerelastic considerations of high
speed flight .
    some structural and aerelastic considerations of high speed flight .   the dominating factors in structural design of high-speed aircraft are thermal and aeroelastic in origin .  the subject matter is concerned largely with a discussion of these factors and their interrelation with one another .  a summary is presented of some of the analytical and experimental tools available to aeronautical engineers to meet the demands of high-speed flight upon aircraft structures .  the state of the art with respect to heat transfer from the boundary layer into the structure, modes of failure under combined load as well as thermal inputs and acrothermoelasticity is discussed .  methods of attacking and alleviating structural and aeroelastic problems of high-speed flight are summarized .  finally, some avenues of fundamental research are suggested ....

=== Generated answer ===
The document does 

---
# Tasks 2–4: Evaluation & Analysis

The remainder of this assignment is about evaluating and analysing the outputs of your RAG system by systematically varying inputs — such as queries, prompt instructions, and retrieval depth — and observing how the outputs change.

## Query Selection

<p style="font-size:1.2em">Manually select queries from the <strong>held-out test set</strong> of your Assignment 2 instruction-tuning data. Do <strong>not</strong> randomly sample from the full Cranfield dataset. If you modify or expand a query (e.g. for better retrieval), you may do so but must describe exactly how you did it. A single query may be reused across <strong>at most two</strong> tasks.</p>

Queries must come from the test set to avoid **data leakage** — if a query was seen during fine-tuning, the model may reproduce a memorised answer rather than reasoning from the retrieved context, making your evaluation unreliable.

---

## Task 2: Generator Evaluation

Compare your two fine-tuned models from Assignment 2 to choose the best generator for your system:
- **RAG A** — fine-tuned with loss over the **full prompt** (instruction + input + response)
- **RAG B** — fine-tuned with loss over the **completion only** (response tokens only)

**Note**: This task evaluates the generator in isolation. Both models receive the same retrieved context. Evaluate the generated responses only, not the quality of the retrieval.

### Implement RAG systems with both the models

In [186]:
# Instantiate one RAGSystem per model using the same BM25 index
rag_A = RAGSystem(model=model_full, tokenizer=tokenizer, bm25=bm25)
rag_B = RAGSystem(model=model_completion, tokenizer=tokenizer, bm25=bm25)

In [187]:
from datasets import load_dataset

test_data = load_dataset("json", data_files={
    "test":  r"../Assignment_2/instruction-tuning dataset/test.jsonl",
})

factual_questions = test_data.filter(lambda x: x['metadata']['template_id'] == 0)['test']
bullet_point_questions = test_data.filter(lambda x: x['metadata']['template_id'] == 3)['test']

In [188]:
factual_question_idxs = [4, 14] 

for i in factual_question_idxs: 
    print(f"=== QUESTION {i} ===")
    #print(factual_questions[i]['input'] + "\n")
    print(factual_questions[i])

selected_factual_questions = [factual_questions[i] for i in factual_question_idxs]

=== QUESTION 4 ===
{'instruction': 'Answer the following technical question using only the information in the document.', 'input': 'Question:\nhow is the heat transfer downstream of the mass transfer region effected\nby mass transfer at the nose of a blunted cone .\n\nDocument:\nthe homogeneous boundary layer at an axisymmetric stagnation\npoint with large rates of injection .\nthe homogeneous boundary layer at an axisymmetric stagnation\npoint with large rates of injection .\n  this report presents a theoretical analysis of the boundary\nlayer at an axisymmetric stagnation point with large rates of\nair injection .  the results of a previous investigation indicated\nthat for localized mass transfer in the stagnation region, the\nrates of injection are considerably greater than those usually\ntreated .  the exact stagnation-point boundary-layer equations\nare integrated numerically for an approximate representation\nof the gas properties .  the two-point boundary conditions are\ntreate

In [189]:
bullet_point_questions_idxs = [17, 25] 

for i in bullet_point_questions_idxs: 
    print(f"=== QUESTION {i} ===")
    #print(bullet_point_questions[i]['input'] + "\n")
    print(bullet_point_questions[i])

selected_bullet_point_questions = [bullet_point_questions[i] for i in bullet_point_questions_idxs]

=== QUESTION 17 ===
{'instruction': 'Using only the document below, answer the question in bullet points.', 'input': 'Question:\nwhat is the criterion for true panel flutter,  as opposed to small\namplitude vibration arising from acoustic disturbances .\n\nDocument:\nfree-flight measurements of the static and dynamic .\nfree-flight measurements of the static and dynamic .\n  representative experimental results are presented to show the\ncurrent status of the panel flutter problem .  results are presented for\nunstiffened rectangular panels and for rectangular panels stiffened by\ncorrugated backing .  flutter boundaries are established for all types\nof panels when considered on the basis of equivalent isotropic plates .\nthe effects of mach number, differential pressure, and aerodynamic\nheating on panel flutter are discussed .  a flutter analysis of\northotropic panels is presented in the appendix .', 'output': '- The provided document does **not** state any criterion for distinguish

In [190]:
import copy

orig_sample = test_data["test"][100]

old_question_block = orig_sample["input"].split("\n\nDocument:")[0]
modified_question = "Question:\nwhat is a rational approximation for the interference effects in the flow field of a wing-body configuration."

modified_sample = copy.deepcopy(orig_sample)
modified_sample["input"] = orig_sample["input"].replace(old_question_block, modified_question)

modified_pair = [orig_sample, modified_sample]

### Part 1: Select Queries

Choose queries covering **all three** conditions below. Fill in your final chosen queries in the code cell (TODO).

| Condition | Minimum | Your queries |
|-----------|---------|---------------|
| Simple factual query | 2 | {'instruction': 'Answer the following technical question using only the information in the document.', 'input': 'Question:\nhow is the heat transfer downstream of the mass transfer region effected\nby mass transfer at the nose of a blunted cone .\n\nDocument:\nthe homogeneous boundary layer at an axisymmetric stagnation\npoint with large rates of injection .\nthe homogeneous boundary layer at an axisymmetric stagnation\npoint with large rates of injection .\n  this report presents a theoretical analysis of the boundary\nlayer at an axisymmetric stagnation point with large rates of\nair injection .  the results of a previous investigation indicated\nthat for localized mass transfer in the stagnation region, the\nrates of injection are considerably greater than those usually\ntreated .  the exact stagnation-point boundary-layer equations\nare integrated numerically for an approximate representation\nof the gas properties .  the two-point boundary conditions are\ntreated in a new manner which is useful for various\nboundary-layer and mixing problems .  the exact solutions indicate that\nfor large rates of injection the boundary layer is closely\nrepresented by an inner isothermal shear flow and by and exterior,\nrelatively thin region, in which the flow variables change to their\nfree-stream values .  an integral method based on profiles\nsuggested by the exact solutions is developed and shown to lead\nto accurate predictions of the integral thicknesses which are\nof interest for a study of the downstream influence of the\nstagnation-point mass transfer .', 'output': 'The provided document does not state how heat transfer downstream of the mass-transfer region is affected by mass transfer at the nose of a blunted cone. It discusses a theoretical analysis of an axisymmetric stagnation-point boundary layer with large air injection and mentions developing integral thickness predictions relevant to studying downstream influence, but it does not give any explicit result about downstream heat transfer on a blunted cone.', 'metadata': {'query_id': '54', 'doc_id': '365', 'template_id': 0, 'source': 'cranfield', 'min_relevance': 3, 'model': 'gpt-5.2', 'temperature': 0.3, 'max_answer_tokens': 180}} <br><br>{'instruction': 'Answer the following technical question using only the information in the document.', 'input': 'Question:\nhas anyone investigated the unsteady lift distributions on finite\nwings in subsonic flow .\n\nDocument:\napproximate indical lift functions for several wings of finite span in\nincompressible flow as obtained from oscillatory lift coefficients .\napproximate indical lift functions for several wings of finite span in\nincompressible flow as obtained from oscillatory lift coefficients .\nthe unsteady-lift functions for a wing undergoing a sudden change in\nsinking speed have been presented for delta wings having aspect ratios\nof 0, 2, and 4 and for rectangular and elliptical wings having aspect\nratios of 0, 3, and 6 . for the elliptical and rectangular wings the\nspanwise lift distributions were also presented . these functions were\ncalculated from the lift coefficients associated with a wing oscillating\n harmonically in pure translational motion, as obtained from several\nsources .\nthe results of these calculations indicate that the normalized\nunsteady-lift functions are substantially independent of the shape of\nthe plan form for elliptical, rectangular, or moderately tapered wings,.\n however, for delta wings the increase of lift toward the steady-state\nvalue is much more rapid than that for the aforementioned wings of the\nsame aspect ratio . these results also corroborate the results of other\ninvestigations in that the rate of growth of lift tends to increase with\n a decrease in aspect ratio . the shape of the spanwise distributions of\n the indicial lift seems to be, for all practical purposes, independent\nof time for rectangular and elliptical wings .', 'output': 'Yes. The document reports investigations of **unsteady-lift functions for finite-span wings in incompressible (subsonic) flow**, including cases where **spanwise (lift) distributions** were presented.\n\n- Unsteady-lift functions for a wing undergoing a **sudden change in sinking speed** were presented for:\n  - **Delta wings** with aspect ratios **0, 2, and 4**\n  - **Rectangular and elliptical wings** with aspect ratios **0, 3, and 6**\n- For the **elliptical and rectangular wings**, the **spanwise lift distributions were also presented**.\n- The document also notes that for **rectangular and elliptical wings**, the **shape of the spanwise distributions of the indicial lift** is, for practical purposes, **independent of time**.', 'metadata': {'query_id': '96', 'doc_id': '699', 'template_id': 0, 'source': 'cranfield', 'min_relevance': 3, 'model': 'gpt-5.2', 'temperature': 0.3, 'max_answer_tokens': 180}}|
| Query requiring a structured/formatted answer (e.g. bullet points) | 2 | {'instruction': 'Using only the document below, answer the question in bullet points.', 'input': 'Question:\nwhat is the criterion for true panel flutter,  as opposed to small\namplitude vibration arising from acoustic disturbances .\n\nDocument:\nfree-flight measurements of the static and dynamic .\nfree-flight measurements of the static and dynamic .\n  representative experimental results are presented to show the\ncurrent status of the panel flutter problem .  results are presented for\nunstiffened rectangular panels and for rectangular panels stiffened by\ncorrugated backing .  flutter boundaries are established for all types\nof panels when considered on the basis of equivalent isotropic plates .\nthe effects of mach number, differential pressure, and aerodynamic\nheating on panel flutter are discussed .  a flutter analysis of\northotropic panels is presented in the appendix .', 'output': '- The provided document does **not** state any criterion for distinguishing **true panel flutter** from **small-amplitude vibration arising from acoustic disturbances**.', 'metadata': {'query_id': '191', 'doc_id': '1008', 'template_id': 3, 'source': 'cranfield', 'min_relevance': 3, 'model': 'gpt-5.2', 'temperature': 0.3, 'max_answer_tokens': 180}} <br> <br> {'instruction': 'Using only the document below, answer the question in bullet points.', 'input': 'Question:\nwhat are the structural and aeroelastic problems associated with flight\nof high speed aircraft .\n\nDocument:\nflutter of rectangular simply supported panels at high\nsupersonic speeds .\nflutter of rectangular simply supported panels at high\nsupersonic speeds .\n  the problem of panel flutter of rectangular simply supported\nplates subjected to supersonic flow over one surface is treated\ntheoretically .  the assumption is made, and subsequently\nverified, that the /static/ approximation to the aerodynamic flutter\nforces yields flutter boundaries with satisfactory accuracy for\nmach numbers greater than about 2 .  two panel flutter analyses\nare performed using this static approximation in conjunction with\nthin-plate theory--one employs aerodynamic strip theory, the\nother aerodynamic surface theory .  the influence of mach\nnumber, dynamic pressure, panel aspect ratio, and midplane\nstress on the panel thickness required to prevent flutter is\ndetermined for extensive ranges of these parameters .', 'output': '- **Panel flutter at high supersonic speeds**\n  - Rectangular, simply supported panels (plates) subjected to **supersonic flow over one surface** can experience **flutter** (an aeroelastic instability).\n\n- **Aerodynamic-force modeling issue (static approximation validity)**\n  - A key aeroelastic/analysis concern is whether the **static approximation** to aerodynamic flutter forces is adequate.\n  - The document states this approximation gives **satisfactory flutter-boundary accuracy for Mach numbers greater than about 2**.\n\n- **Structural sizing problem to prevent flutter**\n  - A structural design problem is determining the **panel thickness required to prevent flutter**, using thin-plate theory coupled with aerodynamic modeling.\n\n- **Parameters influencing flutter and required thickness**\n  - The required thickness (and flutter boundary) is influenced by:\n    - **Mach number**\n    - **Dynamic pressure**', 'metadata': {'query_id': '2', 'doc_id': '391', 'template_id': 3, 'source': 'cranfield', 'min_relevance': 3, 'model': 'gpt-5.2', 'temperature': 0.3, 'max_answer_tokens': 180}}|
| One query tested with two different reformulations (same keywords, different wording / instruction template) | 1 pair | {'instruction': 'Using only the document below, answer the question in bullet points.', 'input': 'Question:\nhow can wing-body,  flow field interference effects be approximated\nrationally .\n\nDocument:\nan experimental and theoretical investigation of second-order supersonic\n wing-body interference, for a non-lifting body with wings at incidence .\nan experimental and theoretical investigation of second-order supersonic\n wing-body interference, for a non-lifting body with wings at incidence .\npressure distributions on the wing of two wing-body combinations are\nmeasured experimentally at mach numbers 3 and 4 with the wing at various\n incidences in the range 0degree to 10degree . the results are compared\nwith theoretical results which include interference effects calculated\naccording to the second-order supersonic wing-body interference theory\ndue to landahl and beane /1/ . this theory, having been tested\npreviously for non-lifting wing-body combinations, is thus tested also for\nwings at incidence . the agreement between theory and experiment is\nfound to vary with mach number and wing sweepback . for the higher mach\nnumber and moderate sweepback the theory gives a good prediction of\npressure distribution, but for the most adverse condition of low mach\nnumber and large sweepback the theory is found to overestimate the\ninterference effects . this is expected as the theory assumes the\nsweepback of the wings is small compared with that of the mach line . an\n empirical guide to the limit of application of the interference theory\nis given . within this limit the agreement between theory and experiment\n is found to deteriorate only a little with increase of incidence, over\nthe range tested .', 'output': '- Use **second-order supersonic wing-body interference theory** to calculate wing-body flow-field interference effects, specifically the theory **due to Landahl and Beane**.  \n- Apply it to configurations including a **non-lifting body with wings at incidence** (tested for incidences **0° to 10°**).  \n- Expect the approximation to be **more reliable at higher Mach numbers (e.g., Mach 4)** and **moderate wing sweepback**, where it gives **good pressure-distribution predictions**.  \n- Recognize a key **assumption/limit**: the theory assumes **wing sweepback is small compared with the Mach-line sweepback**; when this is violated (e.g., **low Mach number (Mach 3) and large sweepback**), it can **overestimate interference effects**.  \n- Use the document’s', 'metadata': {'query_id': '186','doc_id': '1075','template_id': 3,'source': 'cranfield','min_relevance': 3, 'model': 'gpt-5.2', 'temperature': 0.3, 'max_answer_tokens': 180}} <br><br> {'instruction': 'Using only the document below, answer the question in bullet points.', 'input': 'Question:\nwhat is a rational approximation for the interference effects in the flow field of a wing-body configuration .\n\nDocument:\nan experimental and theoretical investigation of second-order supersonic\n wing-body interference, for a non-lifting body with wings at incidence .\nan experimental and theoretical investigation of second-order supersonic\n wing-body interference, for a non-lifting body with wings at incidence .\npressure distributions on the wing of two wing-body combinations are\nmeasured experimentally at mach numbers 3 and 4 with the wing at various\n incidences in the range 0degree to 10degree . the results are compared\nwith theoretical results which include interference effects calculated\naccording to the second-order supersonic wing-body interference theory\ndue to landahl and beane /1/ . this theory, having been tested\npreviously for non-lifting wing-body combinations, is thus tested also for\nwings at incidence . the agreement between theory and experiment is\nfound to vary with mach number and wing sweepback . for the higher mach\nnumber and moderate sweepback the theory gives a good prediction of\npressure distribution, but for the most adverse condition of low mach\nnumber and large sweepback the theory is found to overestimate the\ninterference effects . this is expected as the theory assumes the\nsweepback of the wings is small compared with that of the mach line . an\n empirical guide to the limit of application of the interference theory\nis given . within this limit the agreement between theory and experiment\n is found to deteriorate only a little with increase of incidence, over\nthe range tested .', 'output': '- Use **second-order supersonic wing-body interference theory** to calculate wing-body flow-field interference effects, specifically the theory **due to Landahl and Beane**.  \n- Apply it to configurations including a **non-lifting body with wings at incidence** (tested for incidences **0° to 10°**).  \n- Expect the approximation to be **more reliable at higher Mach numbers (e.g., Mach 4)** and **moderate wing sweepback**, where it gives **good pressure-distribution predictions**.  \n- Recognize a key **assumption/limit**: the theory assumes **wing sweepback is small compared with the Mach-line sweepback**; when this is violated (e.g., **low Mach number (Mach 3) and large sweepback**), it can **overestimate interference effects**.  \n- Use the document’s', 'metadata': {'query_id': '186','doc_id': '1075','template_id': 3,'source': 'cranfield','min_relevance': 3,'model': 'gpt-5.2','temperature': 0.3,'max_answer_tokens': 180}} |


In [195]:
k = 5 # best models evaluated for k=5 in Assignment 2

def pair_testing(sample):
    question = sample["input"].split("\n\nDocument:")[0].replace("Question:\n", "").strip()

    print(f"[id={sample["metadata"]['query_id']}] Question:\n{question}\n")
    print(f"[id={sample["metadata"]['query_id']}] Expected answer:\n{sample["output"]}\n")

    answer_A, docs = rag_A.generate(question, k=k, verbose=True, instruction=sample["instruction"])
    answer_B = rag_B.generate(question, k=k, instruction=sample["instruction"])

    print(f"-> [id={sample["metadata"]['query_id']}] [A] (full-prompt loss):\n{answer_A}\n")
    print(f"-> [id={sample["metadata"]['query_id']}] [B] (completion-only loss):\n{answer_B}\n")
    print(f"-> [docs] {docs}")
    print("=============================\n")

In [196]:
print("\n=== FACTUAL QUERIES ===")
for s in selected_factual_questions:
    pair_testing(s)

print("\n=== STRUCTURED QUERIES ===")
for s in selected_bullet_point_questions:
    pair_testing(s)

print("\n=== REFORMULATION PAIR ===")
for i, sample in enumerate(modified_pair):
    label = "Original" if i == 0 else "Reformulated"
    pair_testing(sample)


=== FACTUAL QUERIES ===
[id=54] Question:
how is the heat transfer downstream of the mass transfer region effected
by mass transfer at the nose of a blunted cone .

[id=54] Expected answer:
The provided document does not state how heat transfer downstream of the mass-transfer region is affected by mass transfer at the nose of a blunted cone. It discusses a theoretical analysis of an axisymmetric stagnation-point boundary layer with large air injection and mentions developing integral thickness predictions relevant to studying downstream influence, but it does not give any explicit result about downstream heat transfer on a blunted cone.

-> [id=54] [A] (full-prompt loss):
The document does not provide information on the effect of mass transfer at the nose of a slender cone on heat transfer downstream of the mass transfer region. It only discusses the effect of mass transfer on shock shape, pressure distribution, and heat transfer in a Mach number of 6 on sharp and blunt 15 cone at ang

### Part 2: Evaluate the outputs of both models using the following criteria where applicable

Use the criteria below to evaluate your outputs and write down your observations.

| Criterion | Description |
|-----------|-------------|
| **Completeness** | Does the answer address the entire query given the available context? |
| **Instruction Following** | Does the model follow the required format / instructions? |
| **Robustness** | Is the model consistent across different reformulations of the same query? |
| **Fluency** | Is the output coherent and readable? |


#### Evaluation Table

TODO: Fill out the observation column on how the models behave. 

| Query type | Model | Observation |
|------------|-------|-------------|
| Factual | Completion-only | Fluent, and well-formed answers that are coherent and readable from a grammatical non content perspective. Howerver,completeness is a weak point: on id=96 the retrieved abstract directly answers the question (delta wings, rectangular/elliptical, spanwise distributions presented) yet the model falsely claims that the document is unrelated. id=54, on the other hand, correctly identifies that the document does not address the downstream-blunted-cone case. |
| | Full prompt | Again fluent and on-template, but there seems to be some context drift. id=54 stitches content from multiple of the k=5 retrieved abstracts (Mach 6 cones, base pressure, wake structure) that the question never asked about. Same false refusal as the completion-only model on id=96. General tendency to summarise the retrieved documents rather than answer the query. |
| Structured | Completion-only | Instruction-following is mixed. id=2 produces a clean bulleted answer with nested sub-bullets that mirrors the expected format. id=191, however, reverts to prose. When the model commits to bullets, formatting and fluency are good and content is grounded in the retrieved abstract. |
| | Full prompt | Complete instruction-following failure for id=191. Loops degeneratively a "Question: … Document: …" output until max_new_tokens is exhausted. Potentially a full-prompt artifact. We can assume that the loss on prompt tokens during training teaches the model to continue the prompt structure rather than transition to a response. On id=2 the output is flat prose with no bullets despite the explicit bullet-point instruction. |
| Reform (v1 vs v2) | Completion-only | Robustness is poor. Retrieved documents and content words are nearly identical across reformulations, but v2 (the modified query) drifts onto a different paper entirely (wing-under-body geometry, Mach 1.6, fineness-ratio optimisation) while v1 (the original test data point) stays on the second-order theory topic. Bullet format is preserved in both, so instruction-following is stable, semantic content, i.e., completeness is not. |
| | Full prompt | Consistent across reformulations but in a degenerate sense. Both versions produce similar prose refusals and ignore the bullet instruction in both. Lower variance, but at the cost of completeness and instruction-following. The model never engages with either phrasing. |

### Part 3: TODO: Based on your evaluations, answer the following questions

**Q1. Is there a query type where one model clearly outperforms the other?**

Yes, on structured, for us "bullet point"-queries. The completion-only model produces a properly bulleted answer on id=2 and a coherent prose fallback on id=191, while the full-prompt model fails completely on id=191 (the above mentioned "Question: … Document: …" loop until max_new_tokens) and ignores the bullet instruction on id=2. That gap is considerable, even though the test sample is same with n=2. On factual queries the two models are roughly tied, both produce fluent, on-template prose, both correctly refuse on id=54, and both incorrectly refuse on id=96. The completion-only model is a bit tighter in its content, but neither model is reliably grounded in the retrieved documents. On the reformulation pair the comparison is difficult since we have two distinctly different yet still 'wrong' answers. Completion-only varies in content across reformulations indicating very poor robustness, while full-prompt is consistent only because it produces the same simple refusal regardless of the query formulation. Neither result is desirable.

**Q2. Which model do you select as the better generator, and what is your justification?**

The completion-only model. First, on the instruction-following criterion that matters significantly for a downstream RAG generator it is the only one of the two that ever produces output in the requested structure. The full-prompt model even enters a complete degenerate loop that would make it unusable in any deployed pipeline. Second, the failure mode describe before is in of itself an inherent consequence of the training objective. With loss applied over the entire prompt, the model is explicitly optimised to predict instruction and input tokens, which means at inference time those tokens are valid continuations of the response. The learned signal that "### Response:" is what is crucial is comparatively much weaker. Completion-only training removes that pitfall for generation, even though there are scenarios, as discussed in Assignment 2 where full prompt can outperform completion only, though unlikely for most RAG systems. Two remaining concerns still persist. Even the completion-only still false-refuses despite a directly relevant retrieved document, and robustness across reformulations is poor. Semantic content shifts even when retrieval is stable (same keywords -> same BM25 retrieval).

**Q3. Does your selection align with the model you expected to perform better based on PPL_resp and PPL_all from Assignment 2? Discuss any discrepancy.**

The selection aligns with PPL_resp (4.72 vs 5.13) but disagrees with PPL_all (17.03 vs 10.27). The qualitative results in Part 2, however, confirm PPL_resp is the right metric for picking a generator. The PPL_all disagreement is a to be expected artifact rather than a real issue. A low PPL_all for the full-prompt model just reflects its training objective. It is optimised to predict instruction and input tokens, which the generator is never asked to produce at inference. That same memorisation is what causes the id=191 failure loop: "Question: … and Document: …" are high-probability continuations for it. Surprisingly, the PPL_resp gap in the evaluation of Assignment 2 is rather small (8%), which doesn't predict how dramatic the qualitative gap is in the evaluation conducted here.


## Task 3: RAG System Evaluation

This task compares your selected model in two settings: with RAG and without retrieval (no-RAG). No-RAG serves as the baseline, and the model receives only the query with instruction and no retrieved context.

- **RAG**: top-*k* Cranfield documents are retrieved and injected into the prompt.
- **No-RAG**: the model answers from parametric knowledge only (no retrieved context).

The goal is to assess how much access to external context improves factual accuracy and reduces hallucination. Use the model you selected at the end of Task 2.

> **Note**: For RAG outputs, inspect the retrieved documents alongside the generated answer. You will need them to evaluate groundedness and use of context.

> **Important**: The no-RAG baseline must use the same `### Instruction` / `### Input` / `### Response` prompt template as the RAG setting, with the retrieved documents omitted. Passing the query as a bare string would turn it into a plain autocompletion prompt, which is not a fair comparison.

In [198]:
# Sample function for generating noRAG responses. You are allowed to edit as you wish 
def generate_no_rag(query: str, model, tokenizer, max_new_tokens: int = 256) -> str:
    """
    Generate an answer using the language model without any retrieved context.

    Serves as the no-RAG baseline: the model receives only the query wrapped
    in the standard instruction-tuning template and must rely entirely on its
    parametric (fine-tuned) knowledge to produce an answer.

    Args:
        query          : raw user question to answer.
        model          : causal language model to use for generation.
        tokenizer      : tokenizer corresponding to *model*.
        max_new_tokens : maximum number of new tokens to generate.

    Returns:
        Decoded answer string with the prompt prefix stripped and leading/
        trailing whitespace removed.
    """
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    prompt = (
        "### Instruction:\nAnswer the following technical question.\n\n"
        f"### Input:\nQuestion: {query}\n\n"
        "### Response:\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    prompt_len = inputs["input_ids"].shape[1]
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(output_ids[0][prompt_len:], skip_special_tokens=True).strip()



### Part 1: Select Queries

Select **at least 5** queries with at least one per condition. Avoid reusing the same set of queries used in Task 2.

| Condition | Description | Your query (TODO)|
|-----------|-------------|------------|
| Simple | Does not require specific domain knowledge |{'instruction': 'Explain the answer to the question below based strictly on the provided document.', 'input': 'Question:\nhas anyone formally determined the influence of joule heating,  produced\nby the induced current,  in magnetohydrodynamic free convection flows\nunder general conditions .\n\nDocument:\nsteady and transient free convection of an electrically\nconducting fluid from a vertical plate in the presence\nof a magnetic field .\nsteady and transient free convection of an electrically\nconducting fluid from a vertical plate in the presence\nof a magnetic field .\n  an analysis is made for the laminar\nfree convection and heat transfer of\na viscous electrically conducting fluid\nfrom a hot vertical plate in the case\nwhen the induced field is negligible\ncompared to the imposed magnetic field .\nit is found that similar solutions for\nvelocity and temperature exist when\nthe imposed magnetic field (acting\nperpendicular to the plate) varies inversely\nas the fourth root of the distance from\nthe lowest end of the plate .  explicit\nexpressions for velocity, temperature,\nboundary layer thickness and nusselt\nnumber are obtained and the effect\nof a magnetic field on them is studied .\nit is found that the effect of the\nmagnetic field is to decrease the rate of\nheat transfer from the wall .  in the\nsecond part, the method of characteristics\nis employed to obtain solutions of\nthe time-dependent hydromagnetic free\nconvection equations (hyperbolic) of\nmomentum and energy put into integral\nform .  the results yield the time required\nfor the steady flow to be established,\nand the effect of the magnetic field on this time is studied .', 'output': 'The provided document does **not** formally determine the influence of **Joule heating produced by the induced current** under general conditions.\n\nWhat it does say is that it analyzes laminar free convection and heat transfer from a hot vertical plate **“when the induced field is negligible compared to the imposed magnetic field.”** That assumption effectively removes (or at least does not address) effects tied to induced-current phenomena under general conditions, including Joule heating due to induced currents. The document focuses instead on how an **imposed transverse magnetic field** affects velocity/temperature similarity solutions, boundary-layer thickness, Nusselt number, and the transient time to reach steady flow.', 'metadata': {'query_id': '20', 'doc_id': '267', 'template_id': 1, 'source': 'cranfield', 'min_relevance': 3, 'model': 'gpt-5.2', 'temperature': 0.3, 'max_answer_tokens': 180}} {'instruction': 'Based on the document, define or describe the concept asked in the question.', 'input': 'Question:\nhas anyone formally determined the influence of joule heating,  produced\nby the induced current,  in magnetohydrodynamic free convection flows\nunder general conditions .\n\nDocument:\nstationary convection flow of an electrically conducting\nliquid between parallel plates in a magnetic field .\nstationary convection flow of an electrically conducting\nliquid between parallel plates in a magnetic field .\n  a study is made of the stationary convection\nof an electrically conducting liquid in the space\nbetween two parallel plates, heated to different\ntemperatures, in the presence of a magnetic\nfield .  the distribution of velocity, temperature,\nand induced fields are found, and the convective\nheat flow is calculated .', 'output': 'The document does not state that anyone has *formally determined* the influence of Joule heating (from induced currents) in magnetohydrodynamic free convection flows **under general conditions**.\n\nIt only describes a specific study of **stationary convection** of an electrically conducting liquid **between two parallel plates** at different temperatures **in a magnetic field**, where the **velocity, temperature, and induced fields** are determined and the **convective heat flow** is calculated. It does not mention Joule heating or a general-condition determination of its influence.', 'metadata': {'query_id': '20', 'doc_id': '407', 'template_id': 4, 'source': 'cranfield', 'min_relevance': 3, 'model': 'gpt-5.2', 'temperature': 0.3, 'max_answer_tokens': 180}}|
| Knowledge-intensive| Requires specific or detailed factual knowledge |{'instruction': 'Explain the answer to the question below based strictly on the provided document.', 'input': 'Question:\nhas anyone formally determined the influence of joule heating,  produced\nby the induced current,  in magnetohydrodynamic free convection flows\nunder general conditions .\n\nDocument:\nsteady and transient free convection of an electrically\nconducting fluid from a vertical plate in the presence\nof a magnetic field .\nsteady and transient free convection of an electrically\nconducting fluid from a vertical plate in the presence\nof a magnetic field .\n  an analysis is made for the laminar\nfree convection and heat transfer of\na viscous electrically conducting fluid\nfrom a hot vertical plate in the case\nwhen the induced field is negligible\ncompared to the imposed magnetic field .\nit is found that similar solutions for\nvelocity and temperature exist when\nthe imposed magnetic field (acting\nperpendicular to the plate) varies inversely\nas the fourth root of the distance from\nthe lowest end of the plate .  explicit\nexpressions for velocity, temperature,\nboundary layer thickness and nusselt\nnumber are obtained and the effect\nof a magnetic field on them is studied .\nit is found that the effect of the\nmagnetic field is to decrease the rate of\nheat transfer from the wall .  in the\nsecond part, the method of characteristics\nis employed to obtain solutions of\nthe time-dependent hydromagnetic free\nconvection equations (hyperbolic) of\nmomentum and energy put into integral\nform .  the results yield the time required\nfor the steady flow to be established,\nand the effect of the magnetic field on this time is studied .', 'output': 'The provided document does **not** formally determine the influence of **Joule heating produced by the induced current** under general conditions.\n\nWhat it does say is that it analyzes laminar free convection and heat transfer from a hot vertical plate **“when the induced field is negligible compared to the imposed magnetic field.”** That assumption effectively removes (or at least does not address) effects tied to induced-current phenomena under general conditions, including Joule heating due to induced currents. The document focuses instead on how an **imposed transverse magnetic field** affects velocity/temperature similarity solutions, boundary-layer thickness, Nusselt number, and the transient time to reach steady flow.', 'metadata': {'query_id': '20', 'doc_id': '267', 'template_id': 1, 'source': 'cranfield', 'min_relevance': 3, 'model': 'gpt-5.2', 'temperature': 0.3, 'max_answer_tokens': 180}} {'instruction': 'Based on the document, define or describe the concept asked in the question.', 'input': 'Question:\nhas anyone formally determined the influence of joule heating,  produced\nby the induced current,  in magnetohydrodynamic free convection flows\nunder general conditions .\n\nDocument:\nstationary convection flow of an electrically conducting\nliquid between parallel plates in a magnetic field .\nstationary convection flow of an electrically conducting\nliquid between parallel plates in a magnetic field .\n  a study is made of the stationary convection\nof an electrically conducting liquid in the space\nbetween two parallel plates, heated to different\ntemperatures, in the presence of a magnetic\nfield .  the distribution of velocity, temperature,\nand induced fields are found, and the convective\nheat flow is calculated .', 'output': 'The document does not state that anyone has *formally determined* the influence of Joule heating (from induced currents) in magnetohydrodynamic free convection flows **under general conditions**.\n\nIt only describes a specific study of **stationary convection** of an electrically conducting liquid **between two parallel plates** at different temperatures **in a magnetic field**, where the **velocity, temperature, and induced fields** are determined and the **convective heat flow** is calculated. It does not mention Joule heating or a general-condition determination of its influence.', 'metadata': {'query_id': '20', 'doc_id': '407', 'template_id': 4, 'source': 'cranfield', 'min_relevance': 3, 'model': 'gpt-5.2', 'temperature': 0.3, 'max_answer_tokens': 180}}|
| Complex / multi-part | Concatenate two questions |{'instruction': 'Answer the following questions using only the information in the documents. The question is two part and are separated by \n\n---\n\n. Answer each individually and irrespective of each other.', 'input': 'Question:\ntheoretical studies of creep buckling .\n\n---\n\nexperimental studies of creep buckling .', 'output': 'The provided document describes a **theoretical/analytical method** for **creep buckling of columns** in terms of estimating how a column’s **allowable load capacity decreases with time under creep**.\n\n- It presents **a method for estimating allowable load capacities** of columns subject to creep. The method uses **approximate stress distributions** that are **derived from isochronous stress–strain curves** to estimate column load capacity.  \n- The document states the method is **conservative for the specific time** for which the estimate is made (i.e., it tends to err on the safe side for that time point).\n- The method is applied to **test data** for **as-received and stabilized 24S-T4 aluminum alloy**, and the comparison between computed and experimental capacities indicates the method is **satisfactory for estimating the decrease in capacity with increasing time\n\n---\n\nThe document notes that experimental tests of creep buckling show **good agreement with Marin’s creep-bending-based theory** **only for columns with large slenderness ratios or large load eccentricities**. It also states that Marin’s analysis **did not account for average axial stress** and **neglected the transient (nonlinear) part of the creep curve**, so it is **not directly applicable to efficient (nearly straight, not overly slender) column designs** where average axial stress is high relative to bending stress.', 'metadata': {'query_id': '132', 'doc_id': '1019', 'template_id': 1, 'source': 'cranfield', 'min_relevance': 3, 'model': 'gpt-5.2', 'temperature': 0.3, 'max_answer_tokens': 180}}|
| Comparative | Ask for similarities or differences between concepts |{'instruction': 'Answer the question below. Your answer must be fully supported by the document and must not include external knowledge.', 'input': "Question:\nhow do kuchemann's and multhopp's methods for calculating lift\ndistributions on swept wings in subsonic flow compare with each other and with\nexperiment .\n\nDocument:\na correlation of airfoil section data with the aerodynamic\nloads measured on a 45 sweptback wing at subsonic mach\nnumbers .\na correlation of airfoil section data with the aerodynamic\nloads measured on a 45 sweptback wing at subsonic mach\nnumbers .\n  an investigation has been made of the possibility of correlating\nairfoil section data with measured pressure distributions over a 45\nsweptback wing in the mach number range from 0.50 to 0.95 at a\nfree-stream reynolds number of approximately 2 million .\nthe wing had an aspect ratio\nof 5.5, a taper ratio of 0.53, naca 64a010 sections normal to the\nquarterchord line, and was mounted on a slender body of revolution .\n  at mach numbers of 0.85 and below, and for wing normal-force\ncoefficients below the maximum normal-force coefficient for an\ninfinite-aspect-ratio wing yawed 45 to the flow (derived from airfoil section\ndata by simple sweep relations), good correlation was obtained over most\nof the wing between wing-section and two-dimensional-airfoil pressure\ndistributions .  for greater normal-force coefficients lateral\nboundary-layer flow permitted the inboard wing sections to rise to high maximum\nsection normal-force coefficients .  the effectiveness of this lateral\nboundary-layer flow disappeared towards the tip .  for all mach numbers,\nthe influence of plan-form effects on the pressure distributions limited\nthe quality of the correlation at the 20- and 95-percent-semispan\nstations .  above a mach number of about 0.85 the shock waves\noriginating at the juncture of the body and the wing trailing edge spread over\nthe span, preventing further application of two-dimensional data .\n  the spanwise load distributions at moderate normal-force coefficients\ncould be predicted from span-loading theory for the entire mach number\nrange of the tests .", 'output': 'The provided document does not mention Kuchemann’s or Multhopp’s methods, so it does not allow a direct comparison between those two calculation methods or between either method and experiment.\n\nWhat the document *does* say about comparison with experiment is:\n\n- For a 45° sweptback wing (AR 5.5, taper 0.53, NACA 64A010 sections) tested at Mach 0.50–0.95, **spanwise load distributions at moderate normal-force coefficients could be predicted from span-loading theory for the entire Mach number range of the tests**.\n- Correlation between wing-section and 2D airfoil pressure distributions was **good over most of the wing** at **Mach ≤ 0.85** and for wing normal-force coefficients below a certain limit derived from 2D data via simple sweep relations; correlation degraded near **', 'metadata': {'query_id': '82', 'doc_id': '205', 'template_id': 5, 'source': 'cranfield', 'min_relevance': 3, 'model': 'gpt-5.2', 'temperature': 0.3, 'max_answer_tokens': 180}} {'instruction': 'Answer the following technical question using only the information in the document.', 'input': "Question:\nhow do kuchemann's and multhopp's methods for calculating lift\ndistributions on swept wings in subsonic flow compare with each other and with\nexperiment .\n\nDocument:\nlow speed tests on 45 sweptback wings .\nlow speed tests on 45 sweptback wings .\n  this report contains the results\nof pressure measurements on three\nand aspect ratio 5, over an\nincidence range up to 10 .  chordwise\nand spanwise lift distributions\nare given, mostly near the centre\nwhere, on two of the wings,\nmodifications had been made to the section\nshape .  it was found that altering\nthe thickness distribution in the\ncentre did not affect the loading but\nthat approximately straight isobars\ncould be obtained at values of\nbelow about 0.1 .  by the incorporation\nof twist and camber in the central\npart the distortion of the lift\ndistribution in the centre could be\navoided at one particular incidence,\nand thus the same chordwise\ndistribution obtained over most of the span .\n  twist and camber alone do not improve\nthe isobar pattern and\ntherefore a thickness modification would be\nneeded to give the desired\nlift distribution and isobar pattern at one\nparticular incidence .\n  the results of experimental investigations\nof the boundary layer\nand of the effect of aspect ratio will be given\nin a later report .", 'output': 'The provided document does not contain any information about **Kuchemenn’s** or **Multhopp’s** methods for calculating lift distributions on swept wings in subsonic flow, nor does it compare those methods with each other or with experiment.\n\nIt only reports **experimental pressure measurements** and resulting **chordwise and spanwise lift distributions** for **45° sweptback wings (aspect ratio 5)** up to **10° incidence**, and discusses how **thickness distribution changes**, and **twist/camber in the central part**, affect the measured loading and isobar patterns.', 'metadata': {'query_id': '82', 'doc_id': '679', 'template_id': 0, 'source': 'cranfield', 'min_relevance': 3, 'model': 'gpt-5.2', 'temperature': 0.3, 'max_answer_tokens': 180}}|
| Out-of-domain | A random query outside the Cranfield aeronautics domain (e.g. What are the components of ibuprofen?) |{'instruction': 'Answer the following technical question using only the information in the document.', 'input': 'Question:\nwhat is the optimal learning rate for a standard Attention is All You Need Transformer.\n\nDocument:', 'metadata': {'query_id': '-1'}}|


In [199]:
simple_sample = test_data.filter(lambda x: x["metadata"]["query_id"] == "90")["test"][0]
print(simple_sample)

knowledge_intensive_sample = test_data.filter(lambda x: x["metadata"]["query_id"] == "20")["test"][0]
print(knowledge_intensive_sample)

complex_sample_part_1 = test_data.filter(lambda x: x["metadata"]["query_id"] == "132")["test"]
complex_sample_part_2 = test_data.filter(lambda x: x["metadata"]["query_id"] == "133")["test"]

s1 = complex_sample_part_1[0]
s2 = complex_sample_part_2[0]

SEP = "\n\n---\n\n"

q1 = s1["input"].split("\n\nDocument:")[0].replace("Question:\n", "")
q2 = s2["input"].split("\n\nDocument:")[0].replace("Question:\n", "")

combined_sample = copy.deepcopy(s1)
combined_sample["input"]  = f"Question:\n{q1}{SEP}{q2}"
combined_sample["output"] = s1["output"] + SEP + s2["output"]

combined_sample["instruction"] = f"Answer the following questions using only the information in the documents. The question is two part and are separated by {SEP}. Answer each individually and irrespective of each other."
print(combined_sample)

comparative_sample = test_data.filter(lambda x: x["metadata"]["query_id"] == "82")["test"][0]
print(comparative_sample)

ood_sample = {
    "instruction": "Answer the following technical question using only the information in the document.",
    "input": "Question:\nwhat is the optimal learning rate for a standard Attention is All You Need Transformer.\n\nDocument:",
    "output": "No expected since OOD",
    "metadata": {"query_id": "-1"},
}
print(ood_sample)

{'instruction': 'Based on the document, define or describe the concept asked in the question.', 'input': 'Question:\nrecent data on shock-induced boundary-layer separation .\n\nDocument:\na method for predicting the onset of buffeting and other separation\neffects from wind tunnel tests on rigid models .\na method for predicting the onset of buffeting and other separation\neffects from wind tunnel tests on rigid models .\nthe method is based on the observation of the divergence that occurs in\nthe variation of mean static pressure at the trailing edge of an\naircraft wing at the critical stage in the development of boundary-layer\nseparation when its influence first spreads to the trailing edge and\nthereby to the overall flow .\nthe significance of the trailing-edge pressure variations and their\nconnection with the effects that separation has on the mean and unsteady\n loads is discussed for various types of separation . good prediction\ncan be obtained from wind-tunnel tests, or war

In [200]:
class NoRAGSystem:
    def __init__(
        self,
        model,
        tokenizer,
        max_new_tokens: int = 500,
    ):
        self.model = model
        self.tokenizer = tokenizer
        self.max_new_tokens = max_new_tokens
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model.to(self.device)

    def build_prompt(self,query):
        return (
            "### Instruction:\nAnswer the following technical question.\n\n"
            f"### Input:\nQuestion: {query}\n\n"
            "### Response:\n"
        )

    def generate(self, query):
        prompt = self.build_prompt(query)
        prompt = self.tokenizer(prompt, return_tensors="pt").to(self.device)
        prompt_len = prompt["input_ids"].shape[1]

        with torch.no_grad():
            answer = self.model.generate(
                    **prompt,
                    max_new_tokens=self.max_new_tokens,
                    pad_token_id=self.tokenizer.eos_token_id,
                )
            
        answer = answer[0, prompt_len:]
        answer = self.tokenizer.decode(answer, skip_special_tokens=True).strip()
        return answer

In [201]:
#TODO: Print out the Queries and Response
rag_system = RAGSystem(model=model_completion, tokenizer=tokenizer, bm25=bm25)
no_rag_system = NoRAGSystem(model=model_completion, tokenizer=tokenizer)

In [202]:
k = 5 # best models evaluated for k=5 in Assignment 2

def rag_v_no_rag_testing(sample):
    question = sample["input"].split("\n\nDocument:")[0].replace("Question:\n", "").strip()

    print(f"[id={sample["metadata"]['query_id']}] Question:\n{question}\n")
    print(f"[id={sample["metadata"]['query_id']}] Expected answer:\n{sample["output"]}\n")

    answer_rag, docs = rag_system.generate(question, k=k, verbose=True, instruction=sample["instruction"])
    answer_no_rag = no_rag_system.generate(question)

    print(f"-> [id={sample["metadata"]['query_id']}] [RAG]:\n{answer_rag}\n")
    print(f"-> [id={sample["metadata"]['query_id']}] [NORAG]:\n{answer_no_rag}\n")
    print(f"-> [docs] {docs}")
    print(f"===========================\n")

print("\n=== SIMPLE QUERY ===")
rag_v_no_rag_testing(simple_sample)

print("\n=== KNOWLEDGE INTENSIVE QUERIES ===")
rag_v_no_rag_testing(knowledge_intensive_sample)

print("\n=== COMPLEX QUERY ===")
rag_v_no_rag_testing(combined_sample)

print("\n=== COMPARATIVE QUERY ===")
rag_v_no_rag_testing(comparative_sample)

print("\n=== OOD QUERY ===")
rag_v_no_rag_testing(ood_sample)



=== SIMPLE QUERY ===
[id=90] Question:
recent data on shock-induced boundary-layer separation .

[id=90] Expected answer:
The document does not provide **recent data** on shock-induced boundary-layer separation. It describes a **method for predicting the onset** of separation effects (including shock-induced separation) using wind-tunnel tests on rigid models.

From the document, **shock-induced boundary-layer separation** is treated as a type of separation whose onset can be predicted by observing a **divergence in the variation of mean static pressure at the wing trailing edge** at a critical stage—specifically when the separation’s influence first **spreads to the trailing edge** and affects the overall flow. The method gives good prediction or warning for **shock-induced separations up to the stage at which the shock wave reaches the trailing edge**. Related indicators include divergences in **wake width, lift coefficient, or shock position**.

-> [id=90] [RAG]:
The document does 

### Part 2: Evaluate Outputs


| Criterion | Description |
|-----------|-------------|
| **Groundedness** | Is the answer supported and constrained by the retrieved context? |
| **Factual Accuracy** | Is the information correct? |
| **Hallucination** | Does the model introduce unsupported or incorrect information? |
| **Refusal** | Does the model acknowledge uncertainty or decline to answer? |
| **Use of Context** | Does the model effectively incorporate relevant retrieved details? |



#### Evaluation Table

TODO: For each query type and setting, write 1–2 concise sentences describing your observations of the model's output. Base your observations on the evaluation criteria listed above. You are free to add extra evaluation criteria.

| Query type | Setting | Observation (TODO) |
|------------|---------|-------------|
| Simple | RAG | id=90 produces a soft refusal that drifts off-topic by inventing "overexpanded conical exhaust nozzles", a phrase nowhere in the query nor cleanly attributable to the abstracts. Further it never engages with the trailing-edge pressure indicator that the retrieved context actually provides. Groundedness and use of context are both weak. The retrieval is essentially wasted.|
| | No-RAG | id=90 falls back on refusal, i.e. it does not specify what kind of data is being asked for, which is conservative but content-free. No hallucination, but no real content either — the model declines rather than commit to anything from parametric knowledge.|
| Knowledge-intensive | RAG |d=20 produces a well-grounded refusal. It correctly states that the retrieved abstract does not formally determine the influence of Joule heating and limits itself to what the document actually covers. Closely matches the expected answer in both stance and scope.|
| | No-RAG |id=20 confidently asserts "Yes, there is a formal determination..." which is a direct hallucination. It then collapses into a degenerative loop repeating "Nonlinear in the sense that..." with no informational content. Strong contrast with the RAG setting. Retrieval appears to be surpressing the hallucination here.|
| Complex | RAG | id=132 exposes an inherent retrieval issue with this two-part construction: the concatenated query carries a less targeted, more mixed set of keywords that make retrieval less precise, while k stays fixed at 5 even though the question complexity has roughly doubled (less than doubled given the topical overlap, but still higher). This shows in the output as partially grounded bullets that recycle the limited retrieved context, repeating the same "Note on creep buckling of columns" header several times before truncation. |
| | No-RAG | id=132 misreads the two-part structure entirely and replies with a meta-comment ("The question is asking for theoretical studies... not experimental studies"), refusing to address either sub-question. Without retrieved context the model has no anchor for either part and produces effectively no substantive content.|
| Comparative | RAG | id=82 correctly identifies that the retrieved abstract does not compare Kuchemann's and Multhopp's methods, but then stitches content from multiple of the k=5 retrieved abstracts (Weissinger method with 61 swept wings, modified strip analysis for flutter) that the question never asked about. The same retrieval-context-drift pattern observed for id=54 in Task 2.|
| | No-RAG |id=82 fabricates a comparison ("they are not applicable to the same problem") and then degenerates into a repeating "Scheffler–Scheffler–..." token loop until max_new_tokens is exhausted. Both factual accuracy and fluency collapse without retrieved context.|
| Out-of-domain | RAG |id=-1 refuses cleanly and anchors the refusal in the (irrelevant) aero abstracts that were retrieved (stiffened cylinders, boundary-layer equations, Crocco/Mangler transformations). The retrieval is clearly off-topic and the model correctly recognises this rather than attempting an answer. This is desirable behaviour for OOD.|
| | No-RAG |id=-1 produces a confidently wrong response ("the optimal learning rate ... is not known. The Transformer architecture is not known to have a fixed learning rate.") and then collapses into a repetition loop of the same sentence. No structured refusal, just a fluent-sounding hallucination that decays into degenerate output.|

### Part 3: TODO: Based on your evaluations, answer the following questions

**Q1. Overall, does RAG improve over no-RAG? Summarize your findings.**

Yes, RAG improves over no-RAG, but the improvement is not really about fine-grained factual accuracy. It is dominated by decoding stability. Three of the five no-RAG outputs (id=20, id=82, id=-1) collapse into full repetition loops with no real content past the first sentence or two, while RAG never enters that regime in our samples. The two no-RAG cases that do not loop (id=90, id=132) produce content-free meta-refusals instead. RAG outputs are readable, mostly on-template, and at least partially grounded in the retrieved context. The improvement is qualified though. On id=90 and id=82 the RAG output still drifts off-question by stitching content from the top-k=5 abstracts that the query never asked about. So groundedness is far from perfect even with retrieval. In short, its more about "RAG make the outputs more coherent, no-RAG has a higher tendency for degenerate outputs", rather than "RAG is correct, no-RAG is wrong" - somewhat in contrast to what i expected initially.


**Q2. For which query types does RAG provide the most noticeable improvement, and why?**

Knowledge-intensive (id=20) and out-of-domain (id=-1). The common thread is that both are queries where the model has no usable parametric knowledge to draw on. Without retrieval it is forced to generate something from priors that do not exist, which is exactly when it confabulates and then loops (id=20 invents "Yes, there is a formal determination ...", id=-1 invents that the optimal learning rate is "not known"). With retrieval, the model has a concrete document to point at and can produce a grounded refusal instead of a fabricated affirmation. So RAG helps most where the no-RAG model is least equipped to answer, which is the regime RAG is actually designed for. The other types improve less because the failure mode is different and RAG itself does not improve the output in these cases.


**Q3. How much does retrieval affect hallucination? Does it consistently reduce hallucinations, or are there cases where it has little or no effect (or even introduces errors)? Discuss with examples.**

Retrieval suppresses the most prominent hallucinations, but does not eliminate them. It also seems to introduce a different category of error in their place. The clearest reductions are id=20 and id=-1, where the no-RAG output makes confident claims with no basis and the RAG output replaces them with a grounded refusal. On id=90 though, the RAG output introduces concepts that are not in the query and (given my limited knowledge) not connected really to the question. id=82 shows the same pattern at larger scale, where the model refuses the asked comparison and then volunteers Weissinger-method content stitched from other top-k abstracts that nobody asked about. So retrieval removes hallucinations but adds retrieval-context drift, where the model basically just summarizes whatever it was given rather than answering the question.

**Q4. How does the overall quality of answers differ between RAG and no-RAG? Consider not just fluency but also substance and filler language.**

RAG outputs appear richer and decoding-stable. No-RAG outputs are generally either degenerate or content-free. Three no-RAG outputs (id=20, id=82, id=-1) collapse into loops. The other two (id=90, id=132) are essentially meta-refusals. Either way no-RAG produces no usable substance. RAG outputs are readable and mostly on-template, but have their own filler tendency. False refusals padded with summaries of retrieved abstracts (id=82, id=90), or the repeated header in id=132. Bottom Line: RAG is generally favorable, but here mostly on stability rather than content quality.


**Q5. How does the model behave on the out-of-domain query with and without RAG? Which behavior is more desirable from a user's perspective? Explain why.**

On id=-1 the two settings diverge sharply. RAG retrieves five irrelevant aero abstracts (stiffened cylinders, boundary-layer equations, Crocco/Mangler transformations) and produces a clean refusal grounded in those documents. No-RAG instead asserts "the optimal learning rate ... is not known. The Transformer architecture is not known to have a fixed learning rate.", while factually wrong (but besides the point) the model also then collapses into a repetition loop. The RAG behaviour is clearly more desirable. The user is told the system has no relevant information, which is true and actionable, and the refusal is anchored in evidence they can inspect. The no-RAG output is overconfident, factually incorrect, and degenerate at the token level all at once.

## Task 4: Error Analysis

Analyse when the RAG pipeline succeeds or fails, and identify where failures originate — retrieval, generation, or both.


### Part 1: Identify One Example per Case

Using open-form queries, find **at least one example** of each case below. For each case, 

(a) determine whether the root cause is retrieval, generation, or both,

(b) propose a solution for the error cases.

| Case | Description | Diagnostic questions |
|------|-------------|----------------------|
| **Refusal** | The model declines to answer or expresses uncertainty | Was the context relevant and sufficient? Did the model fail to use available information? |
| **Incorrect Answer** | The model provides a wrong or unsupported answer | Was the context relevant? Did the model correctly use the retrieved information? |
| **Correct Answer** | The model provides a correct and relevant answer | Was the context relevant? Did the model correctly use the retrieved information? |


In [203]:
#TODO: Print out the Queries and Response
# Find queries that illustrate each case.
# Experiment until you observe each behaviour.

# TAKEN FROM PREVIOUS TASKS
refusal_sample = factual_questions[1] #id 96 from the simple queries 
incorrect_sample = comparative_sample # 
correct_sample = bullet_point_questions[1] #id 2 from the structured queries

#### TODO: Case Analysis

**Refusal case:**
- Query: id=96 — "has anyone investigated the unsteady lift distributions on finite wings in subsonic flow."
- Was the retrieved context relevant and sufficient? Yes. Abstract directly answers it: unsteady-lift functions for delta wings (AR 0/2/4) and rectangular/elliptical wings (AR 0/3/6), with spanwise distributions for the latter.
- Did the model fail to use available information? Yes. False refusal, even mentioning the integral-equation derivation from the abstract while claiming the document does not address the question.
- Root cause (retrieval / generation / both): Generation. Retrieval is correct.
- Proposed solution: Refusal is a "cheap" continuation for the generator. We could add additional grounded-answer-from-context pairs to fine-tuning with explicit NEGATIVES in training as adversarial samples (i.e, refusal even tough context is provided correctly). At inference, we could invest more into the prompt-engineering, aiming to push the model to commit when evidence is available rather than default to refusal.

---

**Incorrect answer case:**
- Query: id=82 — "how do kuchemann's and multhopp's methods for calculating lift distributions on swept wings in subsonic flow compare with each other and with experiment."
- Was the retrieved context relevant and sufficient? Topically related but insufficient. Top-k=5 covers apparently related topics, yet with no mention of Kuchemann or Multhopp.
- Did the model correctly use the retrieved information? Partially. Correctly notes the asked comparison is not in the documents, then drifts and goes on about the Weissinger-method and strip-analysis content we did not asked about.
- Root cause (retrieval / generation / both): Both. BM25 keyword overlap is high enough to retrieve neighbours of the right topic without the right content. Generation then fails to terminate after the refusal and summarises whatever it was given.
- Proposed solution: On retrieval, augment BM25 with a semantic retriever so keyword-similar but content-irrelevant abstracts don't appear as frequently. On generation, preference data where the chosen answer ends at the refusal and the rejected answer pads it with off-topic summary directly targets the drift.

---

**Correct answer case:**
- Query: id=2 — "what are the structural and aeroelastic problems associated with flight of high speed aircraft."
- Was the retrieved context relevant and sufficient? Yes. Abstract directly on-topic: thermal/aeroelastic factors, control-surface flutter, stiffness/damping/mass-balance trade-offs, kinetic heating, jet-efflux damage - don't know exactly what these mean semantically but the documents are "relevant".
- Did the model correctly use the retrieved information? Yes. Properly bulleted answer that extracts content from the abstract without hallucinating beyond it, with the bullet-format instruction respected.
- Root cause (retrieval / generation / both): Both elements succeeded in this case with no immediate need for changes.

### Part 2: Experiment with at least two additional values of $k$

Select one query where retrieval was the cause of failure and one where the generator was the cause. 

Experiment at least two additional values of $k$ for each (one higher and one lower than your original setting) and discuss the results. 



In [204]:
#TODO: Experiment and print out the queries and response. Mention the k values

def k_variation_testing(sample):
    k_low = 1
    k = 5
    k_high = 25

    question = sample["input"].split("\n\nDocument:")[0].replace("Question:\n", "").strip()

    print(f"[id={sample["metadata"]['query_id']}] Question:\n{question}\n")
    print(f"[id={sample["metadata"]['query_id']}] Expected answer:\n{sample["output"]}\n")

    for k_temp in [k_low, k, k_high]:
        answer_B, docs = rag_B.generate(question, k=k_temp, verbose=True, instruction=sample["instruction"]) # model_completion rag system
        print(f"-> k = {k_temp}")
        print(f"-> [docs] {docs}")
        print(f"-> [answer] \n{answer_B}\n")

In [205]:
retrieval_failure_sample = test_data.filter(lambda x: x["metadata"]["query_id"] == "34")["test"][0]
k_variation_testing(retrieval_failure_sample)

[id=34] Question:
have wind tunnel interference effects been investigated on a systematic
basis .

[id=34] Expected answer:
Yes. The document indicates wind tunnel interference effects have been treated systematically: it considers in detail solid blockage, wake blockage, lift effect, and boundary-constraint influences at high Mach number, and provides corrections for various tunnel types (open/closed; rectangular/circular/octagonal), speeds, and 2D/3D flows with multiple aerofoils and wings, along with other interferences (wall boundary layer, static-pressure gradient, working-fluid issues).

-> k = 1
-> [docs] [{'title': 'a study of the simulation of flow with free stream mach number 1 in a\nchoked wind tunnel .', 'text': 'a study of the simulation of flow with free stream mach number 1 in a\nchoked wind tunnel .\nthe degree to which experimental results obtained under choking\nconditions in a wind tunnel with solid walls simulate those associated with\nan unbounded flow with free-st

In [206]:
generation_failure_sample = test_data.filter(lambda x: x["metadata"]["query_id"] == "128")["test"][0]
k_variation_testing(generation_failure_sample)

[id=128] Question:
has anyone programmed a pump design method for a high-speed digital
computer .

[id=128] Expected answer:
The document describes “a rapid approximate method” for designing centrifugal compressors/impellers that was solved numerically (“three numerical solutions… were made”) and reports an “approximate computing time… 15 hours per streamtube.” However, it does **not** explicitly state whether anyone has **programmed** this pump/impeller design method for a **high-speed digital computer**.

-> k = 1
-> [docs] [{'title': 'method for design of pump impellers using a high speed\ndigital computer .', 'text': 'method for design of pump impellers using a high speed\ndigital computer .\n  a method of designing pump impellers\nis derived from the equations of motion\nand continuity for incompressible nonviscous\nrelative flow .  the flow is assumed\nto follow a known stream surface (representing\nblade shape) that extends from hub\nto shroud .  equations are also derived for\n

#### TODO: Discussion

**Does increasing or decreasing $k$ resolve the error, introduce new ones, or make no difference? What does this suggest about the advantages and limitations of retrieval depth?**

For id=128 (generation-cause), k=1 resolves the error. The relevant doc is at rank 1 in all settings, so removing the four topically-similar "false-positives" leaves only the relevant doc and the model produces a correct grounded answer. This isolates the k=5 failure to distractor sensitivity rather than a generic refusal bias. Going to k=25 makes things worse and changes the failure mode: the output no longer concerns pump impellers at all and instead drifts onto the rank-25 document on aerodynamic heating, consistent with last-document anchoring on long context for a 0.5B model.

For id=34 (retrieval-cause), k=25 surfaces the survey paper that the expected answer is built from eventually. The model still refuses though, and the refusal is self-contradictory. It claims the document does not provide information on the topic while in the same response where it uses concrete content from the survey paper. Increasing k therefore fixes retrieval but creates a generation-side problem. k=1 is not informative since the single available doc does not address the question.

The takeaway is that k is not a single global knob. The two failures have opposite requirements: id=128 wants k small to remove the false positives since the k=1 is on point, id=34 wants k large enough to surface a low-ranked relevant doc. k, however, only addresses one stage the failures originate from. id=34 needed the right doc (many retrieved docs since at rank 13) and a generator able to pick this nuance up, id=128 needed the false positives gone. The id=128 k=25 result also shows a upper bound from the generator side. Adding documents past the model's effective (here more assumed) attention horizon actively degrades performance through position effects. So the useful range of k is narrower and more sample-dependent than a single fixed value suggests. A semantic retriever would likely address the document-side issue to some degree without having to train or deploy a larger, more expensive generator.